# SystemVerilog on a verified 4-state cycle interpreter

The SV lane, in one sitting: **write real SystemVerilog in a cell → extract
it to an AST envelope (pyslang) → run it on the verified M0 cycle
interpreter → watch the two things 2-state simulators hide from you — the
x startup and a scheduling race — happen live → check the same designs on
the Lean spec surface.**

The M0 tier (`docs/sv-design-m0.md`): a single module; `always_ff` /
`always @(posedge clk)` / `always_comb` / continuous `assign`; blocking `=`
and nonblocking `<=`; `if`/`else`; 4-state (`0 1 x z`) vectors. The
semantics is **cycle-level with a schedule oracle σ**: each cycle applies
the stimulus, settles combinational logic, runs every posedge process in
the order σ chooses, commits the NBA queue, settles again. `σ` ranges over
exactly the process orderings the LRM leaves open — which is the star of
this notebook.

**Before you start:** run `lake build` once at the repo root (the SV lane
is part of it), and have `python3.12` + pyslang on PATH for the extractor.

In [1]:
import pathlib, sys
root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
            if (p / "lakefile.toml").exists())
sys.path.insert(0, str(root / "tools"))
%load_ext lean_magic

## Step 1 — write SystemVerilog

`%%svfile <name>.sv` writes the cell body to `notebooks/work/<name>.sv` and
runs the SV extractor (`extractors/sv/extract.py`, pyslang-based) on it,
emitting the envelope `<name>.sv.json` (`docs/sv-envelope-schema.md`) with
**elaborated** widths. Like the Python lane, the extractor never fails on
valid input: anything outside the M0 vocabulary becomes an explicit
`Unsupported` node, and the interpreter refuses it loudly rather than
guess. (*Invalid* SV — syntax or semantic errors — is different:
`%%svfile` shows pyslang's diagnostics and fails the cell.) The
summary shows modules, ports, and processes.

The opening design is the gallery's reset counter
(`Examples/counter/counter.sv`, verbatim):

In [2]:
%%svfile counter.sv
module counter (input  logic clk, rst,
                output logic [7:0] count);
  always_ff @(posedge clk)
    if (rst) count <= '0;
    else     count <= count + 8'd1;
endmodule

## Step 2 — run it: the x startup is real

Per the LRM, every 4-state variable without an initializer starts as **x**
— and `x + 1` is x too (arithmetic collapses the whole vector when any bit
is unknown, IEEE 1800 §11.4.3). Two-state simulators quietly start
everything at 0 and hide this; on real silicon and real 4-state simulation,
**until your reset actually arrives, you know nothing**.

`%svrun <design>` runs the working envelope on the M0 interpreter (through
the same `harness/sv/runner.lean` the differential harness uses). `--stim`
takes one JSON object per cycle driving **input ports** (binary strings
with `x`/`z`, or ints rendered at declared width); an input absent from a
cycle holds its previous value. The clock is implicit at cycle level —
every cycle *is* a posedge; the `clk` port itself is just an ordinary input
we drive for the table's readability.

Watch `count` below: two pre-reset cycles of honest x, then reset, then
counting.

In [3]:
%svrun counter --stim '[{"clk": 1, "rst": 0}, {"rst": 0}, {"rst": 1}, {"rst": 0}, {"rst": 0}, {"rst": 0}]'

cycle,clk,rst,count
0,1,0,xxxxxxxx
1,1,0,xxxxxxxx
2,1,1,00000000 (0)
3,1,0,00000001 (1)
4,1,0,00000010 (2)
5,1,0,00000011 (3)


Rows 0–1 are the LRM's startup story: `count = xxxxxxxx` survives the
increment because `x + 1 = x`. Only the reset at cycle 2 establishes a
known state — which is exactly what the refinement theorem at the end of
this notebook says (`counterDesign ⊑@clk[from rst] counterModel`: the model
correspondence starts *at the first sampled reset*, because before it no
abstract state corresponds).

## Step 3 — the race: one design, two legal outcomes

Now the showpiece, from the gallery: two `always @(posedge clk)` blocks
that exchange `a` and `b` with **blocking** assigns. The LRM does not say
which block runs first within the Active region — *both* orders are legal
simulator behavior. `a = 1, b = 2` at time zero (initializers), one clock
edge:

In [4]:
%%svfile race_blk.sv
module race_blk (input logic clk);          // blocking assigns: a race
  logic [7:0] a = 8'd1, b = 8'd2;
  always @(posedge clk) a = b;
  always @(posedge clk) b = a;
endmodule

In [5]:
%svrun race_blk --sigma src --stim '[{"clk": 1}]'
%svrun race_blk --sigma rev --stim '[{"clk": 1}]'

cycle,clk,a,b
0,1,00000010 (2),00000010 (2)


cycle,clk,a,b
0,1,00000001 (1),00000001 (1)


Same design, same stimulus, both schedules legal — and the answers differ:

* **σ_src** (source order — empirically what Xcelium does): `a = b` runs
  first, so `a` becomes 2 and then `b = a` reads the *new* `a` → **(2, 2)**.
* **σ_rev** (reverse order): `b = a` runs first → **(1, 1)**.

A simulator gives you *one* of these and moves on; a regression suite built
on it certifies an accident of scheduling. This is why every theorem in the
SV lane **quantifies over σ**: `d ⊨ P` means *P holds under every legal
schedule*, and for `race_blk` the lane proves the opposite —
`race_blk_race` exhibits two schedules with different traces and
`race_blk_not_deterministic : ¬ Deterministic raceBlkDesign`
(`Examples/race_blk/spec.lean`). The race is not a bug in the interpreter;
it is a *theorem about the design*.

## Step 4 — the fix: nonblocking assignment

Swap the `=` for `<=` and the race disappears. Nonblocking assigns read the
pre-edge state and commit afterwards (the LRM's Active/NBA region split, at
cycle granularity here), so the order of the two blocks cannot matter:

In [6]:
%%svfile swap_nba.sv
module swap_nba (input logic clk);          // nonblocking: correct swap
  logic [7:0] a = 8'd1, b = 8'd2;
  always @(posedge clk) a <= b;
  always @(posedge clk) b <= a;
endmodule

In [7]:
%svrun swap_nba --sigma src --stim '[{"clk": 1}, {}, {}]'
%svrun swap_nba --sigma rev --stim '[{"clk": 1}, {}, {}]'

cycle,clk,a,b
0,1,00000010 (2),00000001 (1)
1,1,00000001 (1),00000010 (2)
2,1,00000010 (2),00000001 (1)


cycle,clk,a,b
0,1,00000010 (2),00000001 (1)
1,1,00000001 (1),00000010 (2)
2,1,00000010 (2),00000001 (1)


Identical traces under both schedules, swapping every cycle — schedule-
**in**dependent. And unlike the simulator's single data point, the lane
proves it for *all* schedules: `swap_nba_swaps` says every posedge step
swaps `a`/`b` under every σ.

## Step 5 — the same designs on the Lean spec surface

`%%lean` cells check Lean code against the build. The SV examples live in
per-example modules (`Examples/<name>/spec.lean` states, `proof.lean`
proves — certified node-for-node equal to the extracted envelopes), and a
`+Module` token on the `%%lean` line imports them. Below:

* `#sv_check` — concrete runs in surface syntax (the non-vacuity
  convention: fixed generous fuel, `x` for all-x columns), reproducing this
  notebook's tables *inside Lean*;
* one `⊨`-form corollary: weakening the proved `swap_nba_swaps` with
  `Models.imp` — no interpreter plumbing, no fuel, no AST in the statement.

In [8]:
%%lean sv_surface +Examples.swap_nba.spec +Examples.race_blk.spec
open LeanModels.Sv
open Examples.swap_nba.proof (swapNbaDesign)
open Examples.race_blk.proof (raceBlkDesign)

-- the showpiece, pinned: (2,2) under source order, (1,1) under reverse
#sv_check raceBlkDesign [[clk := 1]] shows a = [2], b = [2]
#sv_check raceBlkDesign [[clk := 1]] under σ_rev shows a = [1], b = [1]

-- swap_nba's table from above, x startup and all — checked at elaboration
#sv_check swapNbaDesign [[clk := 1], [], []] shows a = [2, 1, 2], b = [1, 2, 1]

/-- A `⊨`-form corollary of the proved `swap_nba_swaps`: under EVERY legal
schedule, each posedge step moves `b`'s value into `a`. -/
theorem swap_nba_a_gets_b :
    swapNbaDesign ⊨ onPosedge fun s s' => SvState.lookup s' "a" = SvState.lookup s "b" :=
  swap_nba_swaps.imp fun _stim _tr h i s s' hs hs' => (h i s s' hs hs').1

## What is actually proved, and where next

The four M0 theorems (surface forms in `Examples/<name>/spec.lean`, raw
proofs in `proof.lean`; all axiom-clean — no `sorry`, no `native_decide`):

1. **`run_functional`** — at a fixed schedule the run judgment
   `d / stim ⇓[σ] tr` is functional: the interpreter is deterministic
   *given* σ. All nondeterminism lives in σ, nowhere else.
2. **`swap_nba_swaps`** — `swapNbaDesign ⊨ onPosedge (swap a b)`: correct
   for **every** legal schedule.
3. **`race_blk_race`** — two legal schedules, same stimulus, different
   traces (the (2,2)/(1,1) you just ran), plus
   `race_blk_not_deterministic`.
4. **`counter_refines`** — `counterDesign ⊑@clk[from rst] counterModel`:
   from the first sampled reset the count column follows the golden model,
   for every schedule and every pre-reset abstract state (the x cycles
   before reset are *outside* the claim, exactly as they should be).

**Checking against a real simulator:** `harness/sv/diff_test.py` runs every
case in `harness/sv/cases.json` on Xcelium (`xrun`) or Icarus and diffs the
canonical per-cycle lines against this interpreter under `σ_src` —
`race_blk` legitimately accepts source *or* reverse order. It needs a
simulator installed, so it is a local tool, not part of the notebook run
(CI without a simulator reports SKIP).

* `docs/sv-spec-surface.md` — the full judgment gallery this surface is
  built toward (`⊨`, `⇓[σ]`, `⊑@clk[from rst]`, `#sv_check`, `sv_prove`).
* `docs/sv-design-m0.md` — the tier contract: 4-state operator semantics
  (Xcelium-verified), the cycle scheduler, and what is deliberately out.
* `Examples/<name>/` — the three-file layout: `<name>.sv`, its envelope,
  `spec.lean` + `proof.lean`.